In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os

In [2]:
paths = ["raw//"+ x for x in os.listdir("raw") if x.endswith('.csv')]

In [3]:
import os

print(os.getcwd())
for p in paths:
    print(p, "->", os.path.exists(p))


c:\Users\mohak\OneDrive\Documents\DATASET\Network_Anomaly_Detection_CICIDS2017
raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Morning.pcap_ISCX.csv -> True
raw//Monday-WorkingHours.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv -> True
raw//Tuesday-WorkingHours.pcap_ISCX.csv -> True
raw//Wednesday-workingHours.pcap_ISCX.csv -> True


In [4]:
paths


['raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Monday-WorkingHours.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
 'raw//Wednesday-workingHours.pcap_ISCX.csv']

In [5]:
test_path = paths[:3]
train_path = ['raw//Monday-WorkingHours.pcap_ISCX.csv',
             'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
             'raw//Wednesday-workingHours.pcap_ISCX.csv']
test_path = test_path[::-1]
test_path

['raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']

In [6]:
columns_list = [pd.read_csv(p,nrows=0).columns.to_list() for p in paths]
  
all_same = all(cols == columns_list[0] for cols in columns_list)
print("All datasets have matching columns:", all_same)

All datasets have matching columns: True


In [7]:
dfs=[pd.read_csv(p) for p in train_path]

merged_df_train = pd.concat(dfs,axis=0,ignore_index=True) 


dfs=[pd.read_csv(p) for p in test_path]

merged_df_test = pd.concat(dfs,axis=0,ignore_index=True) 
print(len(merged_df_train)+len(merged_df_test))

#total should be 2830743 rows

2830743


In [8]:
merged_df_test.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [9]:
# merged_df_test = merged_df_test.drop([' Destination Port'],axis=1)
# merged_df_train = merged_df_train.drop([' Destination Port'],axis=1)

In [10]:
import re

def clean_web_attack_labels(df):
    
    # 1. Clean 'Web Attack - Brute Force'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Brute Force$', 
        'Web Attack - Brute Force', 
        regex=True
    )

    # 2. Clean 'Web Attack - XSS'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*XSS$', 
        'Web Attack - XSS', 
        regex=True
    )
    
    # 3. Clean 'Web Attack - Sql Injection'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Sql Injection$', 
        'Web Attack - Sql Injection', 
        regex=True
    )
    
    return df

merged_df_train = clean_web_attack_labels(merged_df_train.copy())

In [11]:
SMALL_ATTACKS = [
    'Infiltration', 
    'Web Attack - Sql Injection', 
    'Heartbleed',
    'Web Attack - XSS'
]

NEW_LABEL = "Other Attack"
merged_df_train[' Label'] = merged_df_train[' Label'].replace(SMALL_ATTACKS, NEW_LABEL)

In [12]:
merged_df_train[' Label'].value_counts()

 Label
BENIGN                      1858775
DoS Hulk                     231073
DoS GoldenEye                 10293
FTP-Patator                    7938
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Web Attack - Brute Force       1507
Other Attack                    720
Name: count, dtype: int64

In [13]:
merged_df_test[' Label'].value_counts()

 Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64

In [14]:
bot_df = merged_df_test[merged_df_test[" Label"]=="Bot"]
ddos_df = merged_df_test[merged_df_test[" Label"]=="DDoS"]
port_df = merged_df_test[merged_df_test[" Label"]=="PortScan"]
benign_df = merged_df_test[merged_df_test[" Label"]=="BENIGN"]

BOT_TRAIN_FRAC = 0.50
DDOS_TRAIN_FRAC = 0.20
PORT_TRAIN_FRAC = 0.20
RANDOM_SEED = 42

bot_train = bot_df.sample(frac=BOT_TRAIN_FRAC,random_state=RANDOM_SEED)
port_train = port_df.sample(frac=PORT_TRAIN_FRAC,random_state=RANDOM_SEED)
ddos_train = ddos_df.sample(frac=DDOS_TRAIN_FRAC,random_state=RANDOM_SEED)

bot_test = bot_df.drop(bot_train.index)
port_test = port_df.drop(port_train.index)
ddos_test = ddos_df.drop(ddos_train.index)

test_df_final = pd.concat([bot_test,port_test,ddos_test,benign_df],axis=0)
merged_df_train = pd.concat([merged_df_train,bot_train,port_train,ddos_train],axis=0)


In [15]:
RANDOM_SEED = 42

# individual class subsets
hulk_df       = merged_df_train[merged_df_train[" Label"]=="DoS Hulk"]
goldeneye_df  = merged_df_train[merged_df_train[" Label"]=="DoS GoldenEye"]
slowloris_df  = merged_df_train[merged_df_train[" Label"]=="DoS slowloris"]
slowhttp_df   = merged_df_train[merged_df_train[" Label"]=="DoS Slowhttptest"]
ftp_df        = merged_df_train[merged_df_train[" Label"]=="FTP-Patator"]
ssh_df        = merged_df_train[merged_df_train[" Label"]=="SSH-Patator"]
brute_df      = merged_df_train[merged_df_train[" Label"]=="Web Attack - Brute Force"]
other_df      = merged_df_train[merged_df_train[" Label"]=="Other Attack"]
benign_df     = merged_df_train[merged_df_train[" Label"]=="BENIGN"]
bot_df = merged_df_train[merged_df_train[" Label"]=="Bot"]
ddos_df = merged_df_train[merged_df_train[" Label"]=="DDoS"]
port_df = merged_df_train[merged_df_train[" Label"]=="PortScan"]


# 20% samples for test (for missing classes)
hulk_test      = hulk_df.sample(n=6000, random_state=RANDOM_SEED)
goldeneye_test = goldeneye_df.sample(frac=0.2, random_state=RANDOM_SEED)
slowloris_test = slowloris_df.sample(frac=0.2, random_state=RANDOM_SEED)
slowhttp_test  = slowhttp_df.sample(frac=0.2, random_state=RANDOM_SEED)
ftp_test       = ftp_df.sample(frac=0.2, random_state=RANDOM_SEED)
ssh_test       = ssh_df.sample(frac=0.2, random_state=RANDOM_SEED)
brute_test     = brute_df.sample(frac=0.2, random_state=RANDOM_SEED)
other_test     = other_df.sample(frac=0.5, random_state=RANDOM_SEED)

# remaining go back to train
hulk_train      = hulk_df.drop(hulk_test.index)
goldeneye_train = goldeneye_df.drop(goldeneye_test.index)
slowloris_train = slowloris_df.drop(slowloris_test.index)
slowhttp_train  = slowhttp_df.drop(slowhttp_test.index)
ftp_train       = ftp_df.drop(ftp_test.index)
ssh_train       = ssh_df.drop(ssh_test.index)
brute_train     = brute_df.drop(brute_test.index)
other_train     = other_df.drop(other_test.index)




# new test additions
new_test_parts = [
    hulk_test,
    goldeneye_test,
    slowloris_test,
    slowhttp_test,
    ftp_test,
    ssh_test,
    brute_test,
    other_test
]

new_test_df = pd.concat(new_test_parts, axis=0)

# remove these samples from old train
new_train_parts = [
    hulk_train,
    goldeneye_train,
    slowloris_train,
    slowhttp_train,
    ftp_train,
    ssh_train,
    brute_train,
    other_train,
    benign_df,
    bot_df,
    port_df,
    ddos_df
]

new_train_df = pd.concat(new_train_parts, axis=0)
test_df_final  = pd.concat([merged_df_test, new_test_df], axis=0)
merged_df_train = new_train_df   # this is your revised train


In [16]:
merged_df_train[' Label'].value_counts()

 Label
BENIGN                      1858775
DoS Hulk                     225073
PortScan                      31786
DDoS                          25605
DoS GoldenEye                  8234
FTP-Patator                    6350
SSH-Patator                    4718
DoS slowloris                  4637
DoS Slowhttptest               4399
Web Attack - Brute Force       1206
Bot                             983
Other Attack                    360
Name: count, dtype: int64

In [17]:
test_df_final[' Label'].value_counts() 

 Label
BENIGN                      414322
PortScan                    158930
DDoS                        128027
DoS Hulk                      6000
DoS GoldenEye                 2059
Bot                           1966
FTP-Patator                   1588
SSH-Patator                   1179
DoS slowloris                 1159
DoS Slowhttptest              1100
Other Attack                   360
Web Attack - Brute Force       301
Name: count, dtype: int64

In [18]:
TARGET_BENIGN_COUNT = 30000

benign_mask = (merged_df_train[' Label']=="BENIGN")


benign_train_undersampled = merged_df_train[benign_mask].sample(
    n = TARGET_BENIGN_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

attack_mask = (merged_df_train[' Label'] != 'BENIGN')
attack_train_df = merged_df_train[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


train_df_final = pd.concat(
    [attack_train_df,benign_train_undersampled],
    axis=0
)

Sampled BENIGN count: 30,000
Total Attack count preserved: 313,351


In [19]:
#for test df
TARGET_BENIGN_COUNT = 6000

benign_mask = (test_df_final[' Label']=="BENIGN")


benign_train_undersampled = test_df_final[benign_mask].sample(
    n = TARGET_BENIGN_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

attack_mask = (test_df_final[' Label'] != 'BENIGN')
attack_train_df = test_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


test_df_final = pd.concat(
    [attack_train_df,benign_train_undersampled],
    axis=0
)

Sampled BENIGN count: 6,000
Total Attack count preserved: 302,669


In [20]:
#for test df
TARGET_DDOS_COUNT = 6000

ddos_mask = (test_df_final[' Label']=="DDoS")


ddos_train_undersampled = test_df_final[ddos_mask].sample(
    n = TARGET_DDOS_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled DDoS count: {len(ddos_train_undersampled):,}")

attack_mask = (test_df_final[' Label'] != 'DDoS')
attack_train_df = test_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


test_df_final = pd.concat(
    [attack_train_df,ddos_train_undersampled],
    axis=0
)

Sampled DDoS count: 6,000
Total Attack count preserved: 180,642


In [21]:
#for test df
TARGET_PORTSCAN_COUNT = 6000

portscan_mask = (test_df_final[' Label']=="PortScan")


portscan_train_undersampled = test_df_final[portscan_mask].sample(
    n = TARGET_PORTSCAN_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled PortScan count: {len(portscan_train_undersampled):,}")

attack_mask = (test_df_final[' Label'] != 'PortScan')
attack_train_df = test_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


test_df_final = pd.concat(
    [attack_train_df,portscan_train_undersampled],
    axis=0
)

Sampled PortScan count: 6,000
Total Attack count preserved: 27,712


In [22]:
TARGET_DOS_COUNT = 30000

dos_mask = (train_df_final[' Label']=="DoS Hulk")


dos_train_undersampled = train_df_final[dos_mask].sample(
    n = TARGET_DOS_COUNT,
    random_state=RANDOM_SEED
)
print(f"Sampled Dos Hulk count: {len(dos_train_undersampled):,}")

attack_mask = (train_df_final[' Label'] != 'DoS Hulk')
attack_train_df = train_df_final[attack_mask]
print(f"Total Attack count preserved: {len(attack_train_df):,}")


train_df_final = pd.concat(
    [attack_train_df,dos_train_undersampled],
    axis=0
)

Sampled Dos Hulk count: 30,000
Total Attack count preserved: 118,278


In [23]:

print(f"\nFinal balanced training set size: {len(train_df_final):,}")


Final balanced training set size: 148,278


In [24]:
train_df_final[' Label'].value_counts() 

 Label
PortScan                    31786
BENIGN                      30000
DoS Hulk                    30000
DDoS                        25605
DoS GoldenEye                8234
FTP-Patator                  6350
SSH-Patator                  4718
DoS slowloris                4637
DoS Slowhttptest             4399
Web Attack - Brute Force     1206
Bot                           983
Other Attack                  360
Name: count, dtype: int64

In [25]:
test_df_final[' Label'].value_counts() 

 Label
DoS Hulk                    6000
BENIGN                      6000
DDoS                        6000
PortScan                    6000
DoS GoldenEye               2059
Bot                         1966
FTP-Patator                 1588
SSH-Patator                 1179
DoS slowloris               1159
DoS Slowhttptest            1100
Other Attack                 360
Web Attack - Brute Force     301
Name: count, dtype: int64

In [26]:
train_df_final.to_parquet('train_final.parquet')
test_df_final.to_parquet('test_final.parquet')